In [0]:

%pip install yfinance pandas
     

In [0]:
import yfinance as yf
import pandas as pd
import os
from datetime import datetime

# ---------------------------------------------------
# Tickers (UK equities)
# ---------------------------------------------------
tickers = [
    "SHEL.L",  # Shell plc
    "HSBA.L",  # HSBC Holdings plc
    "BP.L",    # BP p.l.c.
    "ULVR.L",  # Unilever PLC
    "VOD.L",   # Vodafone Group Plc
    "BARC.L",  # Barclays PLC
    "AZN.L",   # AstraZeneca PLC
    "SGE.L",   # The Sage Group plc
    "GRG.L",   # Greggs plc
    "BWY.L",   # Bellway plc
    "RSW.L",   # Renishaw plc
    "PNN.L",   # Pennon Group plc
    "SVT.L",   # Severn Trent plc
    "IMI.L",   # IMI plc
    "BBY.L",   # Balfour Beatty plc
    "DRX.L",   # Drax Group plc
    "TEP.L",   # Telecom Plus plc
    "CNA.L",   # Centrica plc
    "NCC.L",   # NCC Group plc
    "SBRY.L"   # J Sainsbury plc
]


# ---------------------------------------------------
# Time partition (snapshot date)
# ---------------------------------------------------
now = datetime.now()
year = now.strftime("%Y")
month = now.strftime("%m")
day = now.strftime("%d")

BASE_PATH = "/Volumes/corporate_data_lakehouse/bronze/raw_comp_data/yfinance"

# ---------------------------------------------------
# Dataset folders
# ---------------------------------------------------
datasets = [
    "income_statement",
    "balance_sheet",
    "cashflow",
    "history",
    "stats"
]

# Create structure
for d in datasets:
    os.makedirs(f"{BASE_PATH}/{d}/{year}/{month}/{day}", exist_ok=True)

# ---------------------------------------------------
# Ingestion loop
# ---------------------------------------------------
for t in tickers:
    print("Processing:", t)

    stock = yf.Ticker(t)

    # ---------------- Income Statement ----------------
    income = stock.financials.T
    income.to_json(
        f"{BASE_PATH}/income_statement/{year}/{month}/{day}/{t}.json",
        orient="index"
    )

    # ---------------- Balance Sheet ----------------
    balance = stock.balance_sheet.T
    balance.to_json(
        f"{BASE_PATH}/balance_sheet/{year}/{month}/{day}/{t}.json",
        orient="index"
    )

    # ---------------- Cashflow ----------------
    cashflow = stock.cashflow.T
    cashflow.to_json(
        f"{BASE_PATH}/cashflow/{year}/{month}/{day}/{t}.json",
        orient="index"
    )

    # ---------------- Price History ----------------
    history = stock.history(period="5y")
    history.to_json(
        f"{BASE_PATH}/history/{year}/{month}/{day}/{t}.json"
    )

    # ---------------- Stats ----------------
    stats = pd.DataFrame([stock.info])
    stats.to_json(
        f"{BASE_PATH}/stats/{year}/{month}/{day}/{t}.json",
        orient="records"
    )

print("\nYFinance ingestion complete")